# पाठ १३ - एजेन्ट स्मृति


## सेटअप

यो नोटबुकले **Microsoft Agent Framework** (MAF) प्रयोग गरेर **दिर्घकालीन स्मृति** सहितको यात्रा बुकिंग एजेन्ट कसरी बनाउने देखाउँछ।

तपाईंले कसरी विभिन्न प्रकारका एजेन्ट स्मृतिहरू — कार्यशील, छोटो अवधिको, र दिर्घकालीन —ले एजेन्टलाई संवादहरूमा जानकारी कसरी राख्ने र प्रयोग गर्ने प्रभाव पार्छन् भनेर सिक्नुहुनेछ।

**आवश्यकताहरू:**
- एक Microsoft Foundry परियोजना जसमा एक तैनात गरिएको च्याट मोडल छ (जस्तै `gpt-4.1-mini`)।
- Azure CLI मा लगइन गरिएको — तपाईंको टर्मिनलमा `az login` चलाउनुहोस्।
- `AZURE_AI_PROJECT_ENDPOINT` — तपाईंको Microsoft Foundry परियोजनाको अन्त्यबिन्दु।
- `AZURE_AI_MODEL_DEPLOYMENT_NAME` — तपाईंको तैनात गरिएको मोडलको नाम।


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity python-dotenv -q

In [ ]:
import logging
logging.getLogger("agent_framework.foundry").setLevel(logging.ERROR)

import os
import json
import dotenv
from typing import Annotated
from datetime import datetime

from agent_framework import tool
from agent_framework.foundry import FoundryChatClient
from azure.identity import DefaultAzureCredential

dotenv.load_dotenv()

endpoint = os.getenv("AZURE_AI_PROJECT_ENDPOINT")
deployment_name = os.getenv("AZURE_AI_MODEL_DEPLOYMENT_NAME")

missing = [k for k, v in {
    "AZURE_AI_PROJECT_ENDPOINT": endpoint,
    "AZURE_AI_MODEL_DEPLOYMENT_NAME": deployment_name
}.items() if not v]

if missing:
    raise ValueError(
        f"Missing required environment variables: {', '.join(missing)}. "
        "Please set them as environment variables (e.g., in your .env file or shell environment)."
    )

In [ ]:
# Create the Microsoft Foundry client
client = FoundryChatClient(
    project_endpoint=endpoint,
    model=deployment_name,
    credential=DefaultAzureCredential()
)

print("Microsoft Foundry client configured")

## एजेन्ट मेमोरीका प्रकारहरू

AI एजेन्टहरूले विभिन्न प्रकारका मेमोरीहरू प्रयोग गर्न सक्छन्, प्रत्येकले फरक उद्देश्य पूरा गर्छ:

### कार्यरत मेमोरी
संवाद थ्रेड स्वयं — एकै सत्रमा आदान-प्रदान गरिएका सन्देशहरू। एजेन्टले सहकार्य कायम राख्नका लागि एउटै थ्रेडका पहिलेका सन्देशहरूलाई फिर्ता हेर्न सक्छ। MAF मा यो **`agent.create_session()`** मार्फत सिर्जना गरिन्छ, जसले `AgentSession` फिर्ता गर्छ।

### छोटो अवधिको मेमोरी
काम वा सत्रको अवधि भर कायम रहने तर स्थायी रूपमा संग्रह नगरिने जानकारी। उदाहरणका लागि, एजेन्ट बहु-पटक योजना बनाउने संवादमा तथ्यहरू सङ्कलन गर्न सक्छ र तिनीहरूलाई अन्तिम यात्रा कार्यक्रम तयार पार्न प्रयोग गर्न सक्छ।

### दीर्घकालीन मेमोरी
प्राथमिकता र तथ्यहरू जुन **सत्रहरू बीच** कायम रहन्छन्। फर्किने प्रयोगकर्ताले आफ्नो आहार प्रतिबन्ध वा यात्रा शैली पुन: दोहोर्याउनु पर्दैन। दीर्घकालीन मेमोरी सामान्यतया बाह्य संग्रहद्वारा समर्थित हुन्छ — डेटाबेस, फाइल, वा भेक्टर इण्डेक्स — र उपकरणमार्फत एजेन्टसम्म प्रस्तुत गरिन्छ।


## सत्रहरूसँग कार्यशील मेमोरी

मेमोरीको सबैभन्दा सरल रूप हो संवाद सत्र। जब तपाईं एउटै सत्र (`agent.create_session()` मार्फत सिर्जना गरिएको) लाई निरन्तर `agent.run()` कलहरूमा पास गर्नुहुन्छ, एजेन्टले त्यो संवादको पुरा इतिहास देख्छ र पहिला भएका विवरणहरू सम्झन सक्दछ।

आइए एउटा यात्रा एजेन्ट बनाउँ र कार्यशील मेमोरी देखाउँ। 


In [ ]:
agent = client.as_agent(
    name="TravelMemoryAgent",
    instructions=(
        "You are a travel agent who remembers user preferences across conversations. "
        "Track destinations mentioned, budget constraints, and travel dates."
    ),
)

session = agent.create_session()

# First message — the user shares preferences
response = await agent.run(
    "I love beach destinations and my budget is $3000",
    session=session,
)
print("Agent:", response)

# Second message — the agent should recall the budget from the thread
response = await agent.run(
    "What did I say my budget was?",
    session=session,
)
print("Agent:", response)

एजेन्टले बजेटलाई सहि तरिकाले सम्झियो किनभने दुवै सन्देशहरू एउटै सत्र साझा गर्छन्। यो **कार्य स्मृति** हो — यो सत्रको जीवनकालको लागि मात्र अस्तित्वमा छ।

### नयाँ थ्रेडसँग के हुन्छ?

यदि हामीले **नयाँ** सत्र सिर्जना गर्छौ भने, एजेन्टले अघिल्लो संवादको कुनै स्मृति हुँदैन:


In [ ]:
new_session = agent.create_session()

response = await agent.run(
    "What is my budget?",
    session=new_session,
)
print("Agent:", response)
print("\n💡 The agent has no memory of the previous conversation — it's a fresh session.")

## दीर्घकालीन स्मृति ढाँचा

प्रयोगकर्ता प्राथमिकताहरू **सत्रहरू बीच** सम्झनका लागि, हामीलाई यस्तो एक सतत संग्रह आवश्यक छ जुन संवाद थ्रेड बाहिर बसोबास गर्छ। एजेन्टले यो संग्रहलाई **टुलहरू** मार्फत पहुँच गर्छ — यिनीहरू कार्यहरू हुन् जसलाई यसले जानकारी बचत र पुनप्राप्तिका लागि कल गर्न सक्छ।

तल हामी एउटा सरल इन-मेमोरी प्राथमिकता संग्रह कार्यान्वयन गर्छौं (उत्पादनमा तपाईं यसलाई डाटाबेस वा भेक्टर अनुक्रमणिका द्वारा समर्थन गर्नुहुन्छ) र यसलाई एजेन्टले प्रयोग गर्न सक्ने टुलहरूमार्फत प्रदर्शन गर्छौं।

### वास्तुकला
```
┌─────────────────┐     ┌──────────────────┐     ┌─────────────────┐
│  MAF Agent      │────▶│  @tool functions  │────▶│  Preference     │
│  (LLM)          │     │  save / retrieve  │     │  Store (dict)   │
└─────────────────┘     └──────────────────┘     └─────────────────┘
         │                                                 │
    AgentSession                                   Persists across
    (working memory)                               sessions
```


In [ ]:
# --- Persistent preference store (simulated) ---
preference_store: dict[str, list[str]] = {}


@tool(approval_mode="never_require")
def save_preference(
    user_id: Annotated[str, "User identifier"],
    preference: Annotated[str, "A travel preference to remember"],
) -> str:
    """Save a user travel preference to long-term memory."""
    preference_store.setdefault(user_id, []).append(preference)
    return f"✅ Stored: {preference}"


@tool(approval_mode="never_require")
def get_preferences(
    user_id: Annotated[str, "User identifier"],
) -> str:
    """Retrieve all saved travel preferences for a user."""
    prefs = preference_store.get(user_id, [])
    if not prefs:
        return f"No saved preferences for {user_id}."
    return "Saved preferences:\n- " + "\n- ".join(prefs)


@tool(approval_mode="never_require")
def search_hotels(
    query: Annotated[str, "Search query — location, amenities, or tags"],
) -> str:
    """Search the hotel database for matching properties."""
    hotels = [
        {"name": "Le Meurice Paris", "location": "Paris, France", "price": 850, "tags": ["luxury", "romantic", "spa"]},
        {"name": "Four Seasons Maui", "location": "Maui, Hawaii", "price": 695, "tags": ["beach", "family", "resort"]},
        {"name": "Aman Tokyo", "location": "Tokyo, Japan", "price": 780, "tags": ["luxury", "city", "spa"]},
        {"name": "Hotel Sacher Vienna", "location": "Vienna, Austria", "price": 420, "tags": ["historic", "accessible", "cultural"]},
        {"name": "Fairmont Whistler", "location": "Whistler, Canada", "price": 380, "tags": ["ski", "family", "mountain"]},
    ]
    q = query.lower()
    matches = [
        h for h in hotels
        if q in h["name"].lower()
        or q in h["location"].lower()
        or any(q in t for t in h["tags"])
    ]
    if not matches:
        matches = hotels[:3]
    return json.dumps(matches, indent=2)


print("✅ Tools defined: save_preference, get_preferences, search_hotels")

### परिदृश्य १ — पहिलो पटक प्रयोगकर्ताले वार्षिकोत्सवको यात्रा बुक गर्छ

सारा पहिलो पटक भ्रमण गर्छिन्। एजेन्टले उपकरणहरू मार्फत उनका प्राथमिकताहरू भण्डारण गर्नुपर्छ र होटलहरू सिफारिस गर्न तिनीहरूलाई प्रयोग गर्नुपर्छ।


In [ ]:
travel_agent = client.as_agent(
    tools=[save_preference, get_preferences],
    name="TravelBookingAssistant",
    instructions=(
        "You are a personalized travel booking assistant with long-term memory.\n"
        "WORKFLOW:\n"
        "1. When a user starts a conversation, call get_preferences() to check for saved information.\n"
        "2. Store any new preferences the user mentions using save_preference().\n"
        "3. Use search_hotels() to find suitable options that match their preferences and budget.\n"
        "4. Do NOT recommend hotels that exceed the user's budget.\n\n"
        "IMPORTANT: Always use user_id='sarah_johnson_123' for all memory operations."
    ),
)

session_1 = travel_agent.create_session()

response = await travel_agent.run(
    "Hi! I'm Sarah and I'm planning a trip for my 10th wedding anniversary. "
    "We love romantic destinations, fine dining, and spa experiences. "
    "My husband has mobility issues, so we need accessible accommodations. "
    "Our budget is around $700-800 per night.",
    session=session_1,
)
print("🤖 Agent:", response)

In [ ]:
response = await travel_agent.run(
    "The Hotel Sacher sounds perfect! We're both vegetarian and I have a "
    "severe nut allergy. Can you note that for future trips?",
    session=session_1,
)
print("🤖 Agent:", response)

In [ ]:
# Verify what was stored
print("📋 Preference store contents:")
for uid, prefs in preference_store.items():
    print(f"\n  User: {uid}")
    for p in prefs:
        print(f"    - {p}")

### परिदृश्य २ — सारा हप्ताहरूपछि फर्किन्छिन्

साराले **नयाँ थ्रेड** सुरु गर्छिन् (नयाँ सत्रको नक्कल गर्दै)। कार्य स्मृति खाली छ, तर दीर्घकालीन प्राथमिकता भण्डारमा अझै उनको जानकारी छ। एजेन्टले यसलाई फर्काउनु पर्छ र सिफारिसहरूलाई व्यक्तिगत बनाउनको लागि प्रयोग गर्नु पर्छ।


In [ ]:
session_2 = travel_agent.create_session()  # New session — no working memory

response = await travel_agent.run(
    "Hi, my husband and I are planning another trip. Can you recommend a good hotel?",
    session=session_2,
)
print("🤖 Agent:", response)
print("\n💡 The agent retrieved Sarah's saved preferences from long-term memory "
      "even though this is a completely new conversation thread.")

In [ ]:
response = await travel_agent.run(
    "Great suggestions! For the Maui option, what activities would you recommend for the kids?",
    session=session_2,
)
print("🤖 Agent:", response)

## सारांश

यस पाठमा तपाईंले एजेन्ट मेमोरीका तीन प्रकारहरू सिक्नुभयो र तिनीहरूलाई Microsoft Agent Framework संग कसरी कार्यान्वयन गर्ने भन्ने कुरा जान्नुभयो:

| मेमोरी प्रकार | MAF मेकानिज्म | आयु अवधि |
|---|---|---|
| **कार्यरत** | `agent.create_session()` | एकल संवाद |
| **अल्पकालीन** | थ्रेड भित्र संचित सन्दर्भ | एकल कार्य / सत्र |
| **दीर्घकालीन** | `@tool` फङ्क्सनहरू मार्फत पहुँच गरिने बाहिरी भण्डार | सत्रहरूमा |

### मुख्य सिकाइहरू
1. **`agent.create_session()`** कार्यरत मेमोरी उपलब्ध गराउँछ — एजेन्टले सत्रभित्र पूर्ण संवाद इतिहास देख्छ।
2. **नयाँ सत्रहरूले सन्दर्भ गुमाउँछन्** — दीर्घकालीन मेमोरी बिना एजेन्टले अघिल्लो संवादहरू सम्झन सक्दैन।
3. **`@tool` फङ्क्सनहरू** अन्तर पूल गर्ने पुल हुन् — तिनीहरूले एजेन्टलाई स्थायी भण्डारबाट जानकारी बचत र पुनः प्राप्त गर्न अनुमति दिन्छ।
4. **व्यक्तिगतकरण समयसँग सुधार हुन्छ** — जति धेरै प्राथमिकताहरू भण्डारण गरिन्छ, एजेन्टका सिफारिसहरू त्यति नै राम्रो हुन्छन्।

### वास्तविक-विश्व अनुप्रयोगहरू
- **ग्राहक सेवा**: ग्राहकको इतिहास र प्राथमिकताहरू सम्झनुहोस्
- **व्यक्तिगत सहायकहरू**: दिन वा हप्ता भरि सन्दर्भ राख्नुहोस्
- **स्वास्थ्य सेवा**: बिरामीको जानकारी र प्राथमिकताहरू ट्र्याक गर्नुहोस्
- **ई-कमर्स**: इतिहासको आधारमा व्यक्तिगत किनमेल

### अर्को कदमहरू
- इन-मेमोरी डिक्सलाई डेटाबेस वा भेक्टर स्टोर (जस्तै Azure AI Search) सँग बदल्नुहोस्
- समय-संवेदी जानकारीको लागि मेमोरीको समाप्ति थप्नुहोस्
- साझा मेमोरी भएको बहु-एजेन्ट प्रणालीहरू निर्माण गर्नुहोस्
- ज्ञान-ग्राफ-समर्थित मेमोरीका लागि Cognee नोटबुक अन्वेषण गर्नुहोस्


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**अस्वीकरण**:
यो दस्तावेज़ AI अनुवाद सेवा [Co-op Translator](https://github.com/Azure/co-op-translator) प्रयोग गरेर अनुवाद गरिएको हो। हामी सही हुन प्रयास गर्छौं, तर कृपया जानकार हुनुस् कि स्वचालित अनुवादमा त्रुटिहरू वा अशुद्धताहरू हुन सक्छन्। मूल दस्तावेज़ यसको मूल भाषामा आधिकारिक स्रोत मानिनुपर्छ। महत्वपूर्ण जानकारीका लागि व्यावसायिक मानव अनुवाद सिफारिस गरिन्छ। यस अनुवादको प्रयोगबाट उत्पन्न कुनै पनि गलत बुझाइ वा त्रुटिको लागि हामी जिम्मेवार छैनौं।
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
